In [1]:
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType, IntegerType, TimestampType
)

# Esquema oficial para gold_embeddings
schema_gold_embeddings = StructType([
    StructField("chunk_id", StringType(), False),         # PK / FK a gold_document_chunks
    StructField("document_id", StringType(), False),      # FK a silver_documents
    StructField("embedding", ArrayType(DoubleType()), False), # Vector numérico denso
    StructField("embedding_model", StringType(), False),  # Nombre del modelo utilizado
    StructField("embedding_dimension", IntegerType(), False), # Dimensión del vector (384)
    StructField("created_at", TimestampType(), False)     # Timestamp de generación
])

# Crear la tabla Delta vacía en la Capa Gold (si no existe)
spark.createDataFrame([], schema_gold_embeddings).write.format("delta").mode("ignore").saveAsTable("gold_embeddings")

print("Tabla 'gold_embeddings' verificada e inicializada en LH_Ecodocs.")

StatementMeta(, 066905bd-820a-4311-ad07-96a41994b89a, 4, Finished, Available, Finished, False)

Tabla 'gold_embeddings' verificada e inicializada en LH_Ecodocs.


In [2]:
import os
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import col
import torch
print(f"PyTorch detectado correctamente. Version: {torch.__version__}")
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# CONFIGURACIÓN DEL MODELO DE EMBEDDINGS (REMEDIADO FABRIC)
# ---------------------------------------------------------
# Usamos el path del modelo compatible con pyTorch / transformers en Spark
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 32

print(f"Cargando el modelo de embeddings '{MODEL_NAME}'...")

try:
    # Intento de carga directa
    embedding_model = SentenceTransformer(MODEL_NAME)
except Exception as e:
    print(f"Aviso: Fallo de carga inicial ({e}). Reintentando con configuracion de fallback...")
    # Carga de respaldo asegurando compatibilidad con HuggingFace Hub en PySpark
    embedding_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={"use_safetensors": False} # Fuerza el uso de pesos binarios tradicionales si Safetensors falla
    )

EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()
print(f"¡Modelo cargado con éxito! Dimensión del vector: {EMBEDDING_DIM}")

# ---------------------------------------------------------
# CONTROL ANTI-RECALCULO (CACHING / INCREMENTAL LOAD)
# ---------------------------------------------------------

# 1. Recuperar los chunk_id que ya existen en gold_embeddings
try:
    existing_embeddings_df = spark.table("gold_embeddings").select("chunk_id")
    existing_chunk_ids = set([r.chunk_id for r in existing_embeddings_df.collect()])
except Exception:
    existing_chunk_ids = set()

print(f"Chunks vectorizados previamente en la base de datos: {len(existing_chunk_ids)}")

# 2. Leer todos los chunks generados en la Fase 6
all_chunks_df = spark.table("gold_document_chunks").collect()

# 3. Filtrar únicamente los chunks pendientes de procesar
pending_chunks = [c for c in all_chunks_df if c.chunk_id not in existing_chunk_ids]

print(f"Chunks pendientes de vectorizar: {len(pending_chunks)}")

# ---------------------------------------------------------
# GENERACIÓN DE EMBEDDINGS POR LOTES
# ---------------------------------------------------------
new_embedding_rows = []

if pending_chunks:
    print(f"Iniciando generación de embeddings en lotes de {BATCH_SIZE}...")
    
    # Extraer textos y metadatos
    chunk_ids = [c.chunk_id for c in pending_chunks]
    doc_ids = [c.document_id for c in pending_chunks]
    texts_to_embed = [c.chunk_text for c in pending_chunks]
    
    # Generar embeddings vectorizados en lotes (Batch processing)
    # convert_to_numpy=True para convertir eficientemente a lista de flotantes
    embeddings_vectors = embedding_model.encode(
        texts_to_embed, 
        batch_size=BATCH_SIZE, 
        show_progress_bar=True, 
        normalize_embeddings=True # Normalización para búsqueda por distancia coseno/producto punto
    )
    
    # Construir filas para PySpark Delta Table
    for i in range(len(pending_chunks)):
        # Convertir vector de numpy a lista nativa de Python con tipo float/double
        vector_list = [float(val) for val in embeddings_vectors[i]]
        
        new_embedding_rows.append(Row(
            chunk_id=chunk_ids[i],
            document_id=doc_ids[i],
            embedding=vector_list,
            embedding_model=MODEL_NAME,
            embedding_dimension=EMBEDDING_DIM,
            created_at=datetime.now()
        ))

# ---------------------------------------------------------
# PERSISTENCIA INCREMENTAL EN DELTA LAKE
# ---------------------------------------------------------
if new_embedding_rows:
    df_new_embeddings = spark.createDataFrame(
        new_embedding_rows, 
        schema=spark.table("gold_embeddings").schema
    )
    
    # Append incremental (preserva los vectores creados en ejecuciones anteriores)
    df_new_embeddings.write.format("delta").mode("append").saveAsTable("gold_embeddings")
    
    print(f" ¡Éxito! Se han generado y guardado {len(new_embedding_rows)} vectores en 'gold_embeddings'.")
else:
    print(" No hay chunks nuevos por procesar. Todos los embeddings ya están calculados y al día.")

StatementMeta(, 066905bd-820a-4311-ad07-96a41994b89a, 5, Finished, Available, Finished, False)

PyTorch detectado correctamente. Version: 2.2.1
Cargando el modelo de embeddings 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

¡Modelo cargado con éxito! Dimensión del vector: 384
Chunks vectorizados previamente en la base de datos: 489
Chunks pendientes de vectorizar: 0
 No hay chunks nuevos por procesar. Todos los embeddings ya están calculados y al día.


StatementMeta(, 066905bd-820a-4311-ad07-96a41994b89a, 7, Finished, Available, Finished, False)

StatementMeta(, 066905bd-820a-4311-ad07-96a41994b89a, 8, Finished, Available, Finished, True)

In [3]:
# Comprobar recuento total y tipos de datos
spark.sql("""
    SELECT 
        embedding_model,
        embedding_dimension,
        COUNT(chunk_id) AS total_vectors,
        MIN(created_at) AS first_gen_date,
        MAX(created_at) AS last_gen_date
    FROM gold_embeddings
    GROUP BY embedding_model, embedding_dimension
""").show(truncate=False)

# Inspeccionar la estructura de un vector (verificar que no haya NULLs y la longitud coincida con 384)
spark.sql("""
    SELECT 
        chunk_id, 
        document_id, 
        embedding_model, 
        embedding_dimension, 
        SIZE(embedding) AS vector_length,
        SLICE(embedding, 1, 5) AS first_5_dimensions
    FROM gold_embeddings
    LIMIT 5
""").show(truncate=False)

StatementMeta(, 066905bd-820a-4311-ad07-96a41994b89a, 6, Finished, Available, Finished, False)

+-----------------------------------------------------------+-------------------+-------------+--------------------------+--------------------------+
|embedding_model                                            |embedding_dimension|total_vectors|first_gen_date            |last_gen_date             |
+-----------------------------------------------------------+-------------------+-------------+--------------------------+--------------------------+
|sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2|384                |489          |2026-08-11 10:16:26.469871|2026-08-11 10:16:26.492334|
+-----------------------------------------------------------+-------------------+-------------+--------------------------+--------------------------+

+------------+--------------------------------+-----------------------------------------------------------+-------------------+-------------+------------------------------------------------------------------------------------------------------------